In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    log_loss, roc_auc_score, brier_score_loss, f1_score,
    recall_score, confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[warning] xgboost not installed — skipping XGBoost.")

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"
NUMERIC_FEATURES = ["easting", "northing", "disturb", "rainann",
                     "soildepth", "soilfert", "tempann", "topo"]
CLIP = 1e-6


def build_models():
    models = {
        "LogisticRegression": lambda: LogisticRegression(max_iter=2000, class_weight="balanced"),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, class_weight="balanced",
            random_state=42, n_jobs=-1),
        "MLP": lambda: MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3,
                                      max_iter=1500, random_state=42),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
    return models


def maybe_set_xgb_weight(model, y_train):
    if HAS_XGB and isinstance(model, XGBClassifier):
        pos, neg = max(y_train.sum(), 1), max(len(y_train) - y_train.sum(), 1)
        model.set_params(scale_pos_weight=neg / pos)


def clipped_log_loss(y_true, y_pred):
    p = np.clip(y_pred, CLIP, 1 - CLIP)
    return log_loss(y_true, p, labels=[0, 1])


def safe_auc(y_true, y_pred):
    return roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan


def best_f1_threshold(y_train_true, y_train_pred):
    """Grid search a probability threshold on TRAINING predictions (not
    test) to maximise F1, given severe class imbalance."""
    thresholds = np.linspace(0.01, 0.9, 90)
    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        calls = (y_train_pred >= t).astype(int)
        f1 = f1_score(y_train_true, calls, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


def compute_binary_metrics(y_true, y_pred_proba, threshold):
    calls = (y_pred_proba >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        return {"f1": np.nan, "sensitivity": np.nan, "specificity": np.nan}
    tn, fp, fn, tp = confusion_matrix(y_true, calls, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = f1_score(y_true, calls, zero_division=0)
    return {"f1": f1, "sensitivity": sensitivity, "specificity": specificity}


def full_metrics(y_true, p_test, y_train_true, p_train):
    t_opt = best_f1_threshold(y_train_true, p_train)
    metrics = {
        "log_loss": clipped_log_loss(y_true, p_test),
        "auc": safe_auc(y_true, p_test),
        "brier": brier_score_loss(y_true, p_test),
        "threshold_used": t_opt,
    }
    metrics.update({f"{k}_at_0.5": v for k, v in
                     compute_binary_metrics(y_true, p_test, 0.5).items()})
    metrics.update({f"{k}_at_opt": v for k, v in
                     compute_binary_metrics(y_true, p_test, t_opt).items()})
    return metrics


# ----------------------------------------------------------------------
# APPROACHES
# ----------------------------------------------------------------------
def run_single(train, test, model_fn):
    rows = []
    for species, train_sp in train.groupby(SPECIES_COL):
        test_sp = test.loc[test[SPECIES_COL] == species]
        if test_sp.empty:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)

        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_train, y_train)
        p_test = pipe.predict_proba(X_test)[:, 1]
        p_train = pipe.predict_proba(X_train)[:, 1]

        m = full_metrics(y_test, p_test, y_train, p_train)
        m["species"] = species
        m["approach"] = "single"
        rows.append(m)
    return pd.DataFrame(rows)


def run_multi(train, test, model_fn):
    X_train = train[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train = train[TARGET].astype(int)
    X_test = test[NUMERIC_FEATURES + [SPECIES_COL]]

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [SPECIES_COL]),
    ])
    model = model_fn()
    maybe_set_xgb_weight(model, y_train)
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
    p_test_all = pipe.predict_proba(X_test)[:, 1]
    p_train_all = pipe.predict_proba(X_train)[:, 1]

    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask_test = (test[SPECIES_COL] == species).values
        mask_train = (train[SPECIES_COL] == species).values
        y_test_sp = test_sp[TARGET].astype(int)
        y_train_sp = train.loc[mask_train, TARGET].astype(int)
        if y_train_sp.empty:
            continue
        m = full_metrics(y_test_sp, p_test_all[mask_test], y_train_sp, p_train_all[mask_train])
        m["species"] = species
        m["approach"] = "multi"
        rows.append(m)
    return pd.DataFrame(rows)


def plot_diff_heatmap(pivot_diff, title, path, cbar_label):
    fig, ax = plt.subplots(figsize=(9, 5))
    vmax = np.nanmax(np.abs(pivot_diff.values))
    im = ax.imshow(pivot_diff.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot_diff.columns)))
    ax.set_xticklabels(pivot_diff.columns)
    ax.set_yticks(range(len(pivot_diff.index)))
    ax.set_yticklabels(pivot_diff.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=cbar_label)
    for i in range(pivot_diff.shape[0]):
        for j in range(pivot_diff.shape[1]):
            val = pivot_diff.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        color="black", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    models = build_models()

    all_rows = []
    for model_name, model_fn in models.items():
        print(f"Running {model_name} ...")
        single_df = run_single(train, test, model_fn)
        single_df["model"] = model_name
        multi_df = run_multi(train, test, model_fn)
        multi_df["model"] = model_name
        all_rows.append(single_df)
        all_rows.append(multi_df)

    full = pd.concat(all_rows, ignore_index=True)
    full_path = os.path.join(OUTPUT_DIR, "full_metrics_per_species.csv")
    full.to_csv(full_path, index=False)
    print(f"[saved] {full_path}")

    # ---- heatmaps: rows = species, cols = model, value = single - multi ----
    for metric_col, label, fname in [
        ("log_loss", "log-loss diff (single - multi)", "heatmap_logloss_diff.png"),
        ("f1_at_opt", "F1 diff (single - multi) at train-optimal threshold", "heatmap_f1_diff.png"),
    ]:
        wide = full.pivot_table(index="species", columns=["model", "approach"], values=metric_col)
        diff_cols = {}
        for model_name in models.keys():
            if (model_name, "single") in wide.columns and (model_name, "multi") in wide.columns:
                diff_cols[model_name] = wide[(model_name, "single")] - wide[(model_name, "multi")]
        diff_df = pd.DataFrame(diff_cols)
        plot_diff_heatmap(diff_df, f"{label}\n(blue = multi wins, red = single wins)",
                           os.path.join(OUTPUT_DIR, fname), label)

    # ---- overall (population-weighted) summary ----
    overall_rows = []
    for model_name in models.keys():
        for approach, df in [("single", full[(full.model == model_name) & (full.approach == "single")]),
                              ("multi", full[(full.model == model_name) & (full.approach == "multi")])]:
            overall_rows.append({
                "model": model_name, "approach": approach,
                "mean_log_loss": df["log_loss"].mean(),
                "mean_auc": df["auc"].mean(),
                "mean_brier": df["brier"].mean(),
                "mean_f1_at_opt": df["f1_at_opt"].mean(),
                "mean_sensitivity_at_opt": df["sensitivity_at_opt"].mean(),
                "mean_specificity_at_opt": df["specificity_at_opt"].mean(),
            })
    summary_df = pd.DataFrame(overall_rows)
    summary_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[saved] {summary_path}")

    print("\n===== OVERALL METRICS SUMMARY (mean across species) =====")
    print(summary_df.to_string(index=False))


if __name__ == "__main__":
    main()

Running LogisticRegression ...
Running RandomForest ...
Running MLP ...
Running XGBoost ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/full_metrics_per_species.csv
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/heatmap_logloss_diff.png
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/heatmap_f1_diff.png
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/metrics_summary.csv

===== OVERALL METRICS SUMMARY (mean across species) =====
             model approach  mean_log_loss  mean_auc  mean_brier  mean_f1_at_opt  mean_sensitivity_at_opt  mean_specificity_at_opt
LogisticRegression   single       0.445847  0.758139    0.146575        0.229961                 0.391188                 0.871948
LogisticRegression    multi       0.519727  0.668834    0.173044        0.205694                 0.575492                 0.666197
      RandomFo